# 🏥 Production-Grade Corrective RAG (CRAG) for Medical QA
> **LangGraph + Gemini + ChromaDB + Sentence Transformers**

---
## Architecture Overview
```
Query → RETRIEVE → GRADE DOCS → [RELEVANT?]
                                    ├─ ALL RELEVANT  → GENERATE → HALLUCINATION CHECK → Answer
                                    ├─ SOME IRRELEVANT → TRANSFORM QUERY → RE-RETRIEVE → GRADE → GENERATE
                                    └─ ALL IRRELEVANT → WEB SEARCH FALLBACK → GENERATE
```

## 📦 Section 1: Environment Setup & Imports

In [ ]:
# Install required packages
# !pip install langchain langchain-google-genai langgraph chromadb
# !pip install sentence-transformers tavily-python langsmith
# !pip install pypdf tiktoken pandas numpy scikit-learn
# !pip install langchain-community langchain-chroma

In [ ]:
import os
import json
import re
import uuid
import warnings
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Optional, Tuple, Literal
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# LangChain core
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

# LangChain integrations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.tools.tavily_search import TavilySearchResults

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.graph import CompiledGraph

# Typing for state
from typing_extensions import TypedDict

# Sklearn for cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

# LangSmith tracing
from langsmith import Client as LangSmithClient

print('✅ All imports successful')

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — fill in your API keys here
# ─────────────────────────────────────────────
import getpass

# Google Gemini
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY') or getpass.getpass('Enter Google API Key: ')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# Tavily (web search fallback)
TAVILY_API_KEY = os.environ.get('TAVILY_API_KEY') or getpass.getpass('Enter Tavily API Key: ')
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

# LangSmith (optional — set to empty string to disable)
LANGSMITH_API_KEY = os.environ.get('LANGSMITH_API_KEY', '')
if LANGSMITH_API_KEY:
    os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_PROJECT'] = 'Medical-CRAG'
    print('✅ LangSmith tracing enabled')
else:
    print('ℹ️  LangSmith tracing disabled (no API key)')

# ChromaDB persistence directory
CHROMA_PERSIST_DIR = './chroma_medical_db'
COLLECTION_NAME = 'medical_documents'

# Chunking config
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
RETRIEVAL_K = 5

print('✅ Configuration loaded')

## 📄 Section 2: Data Ingestion & Chunking

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# SYNTHETIC MEDICAL CORPUS
# In production, replace with: PyPDFLoader('path/to/guidelines.pdf')
# ─────────────────────────────────────────────────────────────────────

MEDICAL_CORPUS = [
    # ── Drug References ──
    {
        'title': 'Metformin Clinical Reference',
        'content': """
        Metformin (Glucophage) is a biguanide antidiabetic agent used as first-line therapy for type 2 diabetes mellitus.
        Standard adult dosage: Initial dose 500 mg twice daily or 850 mg once daily with meals.
        Maximum daily dose: 2550 mg/day. Dose titration should occur over 2-3 weeks to minimize GI side effects.
        Contraindications: eGFR < 30 mL/min/1.73m², acute/chronic metabolic acidosis, hypersensitivity to metformin.
        Caution: Hold 48 hours before and after iodinated contrast administration.
        Common side effects: nausea, vomiting, diarrhea, abdominal pain (GI effects often transient).
        Rare but serious: Lactic acidosis (risk increased in renal/hepatic impairment, alcohol abuse).
        Mechanism: Decreases hepatic glucose production, increases insulin sensitivity, reduces intestinal glucose absorption.
        Monitoring: Renal function (eGFR) at baseline and annually; Vitamin B12 levels every 2-3 years.
        Drug interactions: Carbonic anhydrase inhibitors increase lactic acidosis risk. Cimetidine increases metformin plasma levels.
        Pregnancy category: Generally avoided in first trimester; use in second/third trimester under close monitoring.
        """,
        'category': 'drug_reference'
    },
    {
        'title': 'Lisinopril ACE Inhibitor Reference',
        'content': """
        Lisinopril is an angiotensin-converting enzyme (ACE) inhibitor used in hypertension, heart failure, and post-MI management.
        Hypertension dosing: Initial 10 mg once daily; maintenance 20-40 mg once daily; maximum 80 mg/day.
        Heart failure dosing: Initial 2.5-5 mg once daily; target dose 20-40 mg once daily.
        Post-MI dosing: 5 mg within 24 hours, then 5 mg after 24 hours, then 10 mg after 48 hours, then 10 mg daily.
        Contraindications: History of angioedema with ACE inhibitors, bilateral renal artery stenosis, pregnancy (all trimesters).
        Black box warning: Can cause fetal/neonatal morbidity and death. Discontinue immediately if pregnancy detected.
        Side effects: Dry persistent cough (10-15% of patients), hyperkalemia, hypotension (first dose), angioedema (rare but serious).
        Monitoring: Serum creatinine, potassium, blood pressure within 1-2 weeks of initiation and after dose changes.
        Renal dosing: CrCl 10-30 mL/min: initial 5 mg/day; CrCl < 10 mL/min: initial 2.5 mg/day.
        Drug interactions: NSAIDs reduce antihypertensive effect; potassium-sparing diuretics increase hyperkalemia risk.
        """,
        'category': 'drug_reference'
    },
    {
        'title': 'Warfarin Anticoagulation Protocol',
        'content': """
        Warfarin is a vitamin K antagonist anticoagulant used for prevention and treatment of thromboembolic disorders.
        Indications: Atrial fibrillation, DVT/PE treatment and prevention, mechanical heart valves, hypercoagulable states.
        Target INR ranges: Atrial fibrillation: 2.0-3.0; Mechanical mitral valve: 2.5-3.5; DVT/PE: 2.0-3.0.
        Initiation: Loading dose 5-10 mg day 1-2 (use 2.5-5 mg in elderly, small patients, or hepatic impairment).
        Monitoring: INR checked after 2-3 days then adjusted; once stable, monthly INR checks.
        Dose adjustment: If INR 3.0-3.5 (target 2-3): reduce weekly dose by 5-10%. If INR > 4.0: hold 1-2 doses.
        Reversal agents: Vitamin K (oral/IV), Fresh Frozen Plasma, Prothrombin Complex Concentrate (PCC), Phytonadione.
        Drug interactions (extensive): Increases anticoagulant effect: amiodarone, fluconazole, metronidazole, aspirin.
        Decreases anticoagulant effect: rifampin, carbamazepine, vitamin K-rich foods.
        Diet counseling: Maintain consistent vitamin K intake; avoid large variations in leafy green vegetables.
        Pregnancy: Contraindicated in first trimester (embryopathy); use heparin as alternative.
        """,
        'category': 'drug_reference'
    },
    {
        'title': 'Amoxicillin Antibiotic Reference',
        'content': """
        Amoxicillin is a broad-spectrum aminopenicillin antibiotic effective against gram-positive and some gram-negative organisms.
        Adult dosage (standard infections): 500 mg every 8 hours or 875 mg every 12 hours.
        Severe infections: 875 mg every 8 hours or 1000 mg every 12 hours.
        Pediatric dosing: 25-45 mg/kg/day divided every 8-12 hours (maximum 90 mg/kg/day for AOM).
        Community-acquired pneumonia: 1000 mg every 8 hours (adults), 90 mg/kg/day for children.
        H. pylori eradication: Part of triple therapy — amoxicillin 1000 mg + clarithromycin 500 mg + PPI twice daily × 14 days.
        Strep pharyngitis: 500 mg twice daily × 10 days; preferred over penicillin due to tolerability.
        Contraindications: Hypersensitivity to penicillins; use with caution in cephalosporin allergy (cross-reactivity ~1-2%).
        Renal adjustment: CrCl 10-30: every 12 hours; CrCl < 10: every 24 hours.
        Side effects: Diarrhea, nausea, rash (maculopapular rash in patients with EBV infection).
        """,
        'category': 'drug_reference'
    },
    {
        'title': 'Atorvastatin Lipid Management Reference',
        'content': """
        Atorvastatin (Lipitor) is an HMG-CoA reductase inhibitor (statin) for management of hyperlipidemia and cardiovascular risk.
        Intensity classification: Low: 10 mg; Moderate: 10-20 mg; High: 40-80 mg daily.
        ACC/AHA High-intensity statin indication: ASCVD risk ≥7.5%, LDL ≥190 mg/dL, diabetes age 40-75.
        Target LDL reduction: High-intensity atorvastatin reduces LDL by ≥50%; moderate by 30-50%.
        Maximum dose: 80 mg/day (associated with increased myopathy risk).
        Contraindications: Active liver disease, pregnancy/lactation, unexplained elevated transaminases.
        Side effects: Myalgia (5-10%), elevated liver enzymes, new-onset diabetes (modest risk), cognitive effects (rare).
        Serious adverse effects: Rhabdomyolysis (rare; risk increased with interacting drugs).
        Drug interactions: Cyclosporine, gemfibrozil, niacin, antifungals increase myopathy risk.
        Monitoring: Baseline LFTs; CK if symptoms of myopathy; lipid panel at 4-12 weeks after initiation.
        Administration: Can be taken at any time of day without regard to meals.
        """,
        'category': 'drug_reference'
    },
    # ── Clinical Guidelines ──
    {
        'title': 'Sepsis Management Guidelines (Surviving Sepsis Campaign)',
        'content': """
        Sepsis is defined as life-threatening organ dysfunction caused by dysregulated host response to infection.
        Septic shock: sepsis with circulatory, cellular, and metabolic dysfunction; vasopressors needed to maintain MAP ≥65 mmHg.
        Hour-1 Bundle (mandatory within 1 hour of recognition):
        1. Measure lactate; re-measure if initial lactate > 2 mmol/L
        2. Obtain blood cultures before antibiotic administration
        3. Administer broad-spectrum antibiotics
        4. Begin 30 mL/kg crystalloid for hypotension or lactate ≥ 4 mmol/L
        5. Apply vasopressors if hypotensive during/after fluid resuscitation to maintain MAP ≥ 65 mmHg
        Antibiotic selection: Broad-spectrum coverage of likely pathogens; de-escalate based on cultures/sensitivity.
        Fluid resuscitation: Target MAP ≥65 mmHg; use dynamic measures (pulse pressure variation, PLR) to guide further fluids.
        Vasopressors: Norepinephrine is first-line; add vasopressin 0.03 units/min or epinephrine if additional agent needed.
        Corticosteroids: IV hydrocortisone 200 mg/day if hemodynamically unstable despite fluids and vasopressors.
        Glucose management: Target blood glucose 140-180 mg/dL using insulin protocol.
        Ventilation: Low tidal volume strategy (6 mL/kg IBW) if ARDS present; prone positioning if PaO2/FiO2 < 150.
        Monitoring: Repeat lactate every 2 hours until < 2 mmol/L; hourly urine output; continuous cardiac monitoring.
        """,
        'category': 'clinical_guideline'
    },
    {
        'title': 'Hypertension Management Guidelines (JNC 8 / ACC/AHA 2017)',
        'content': """
        Hypertension classification (ACC/AHA 2017): Normal: < 120/< 80; Elevated: 120-129/< 80;
        Stage 1 HTN: 130-139/80-89; Stage 2 HTN: ≥140/≥90; Hypertensive crisis: >180/>120.
        Treatment thresholds: Start medication at ≥130/80 with CVD or 10-year ASCVD risk ≥10%.
        For low-risk patients: Lifestyle modification first; initiate medication at ≥140/90.
        First-line drug classes: Thiazide diuretics (chlorthalidone preferred), ACE inhibitors/ARBs, dihydropyridine CCBs.
        Compelling indications:
        - Diabetes: ACE inhibitor or ARB (renal protective)
        - CKD with proteinuria: ACE inhibitor or ARB
        - Post-MI: Beta-blocker + ACE inhibitor
        - Heart failure: ACE inhibitor/ARB + beta-blocker + diuretic + aldosterone antagonist
        Blood pressure targets: General: < 130/80; Very elderly (≥80): < 150/80 acceptable.
        Lifestyle modifications: Weight loss (1 mmHg per kg lost), DASH diet, sodium restriction (< 2.4 g/day),
        physical activity (150 min/week moderate intensity), limit alcohol (≤1 drink/day women, ≤2 men).
        Resistant hypertension: On ≥3 drugs at maximally tolerated doses; add spironolactone or consider renal denervation.
        Hypertensive urgency: Oral antihypertensives; reduce BP by 25% over 24 hours; no rapid reduction needed.
        Hypertensive emergency: IV labetalol, nicardipine, or hydralazine; reduce MAP by 10-20% in first hour.
        """,
        'category': 'clinical_guideline'
    },
    {
        'title': 'Type 2 Diabetes Management Protocol',
        'content': """
        Type 2 diabetes management follows a stepwise approach guided by HbA1c targets and patient comorbidities.
        HbA1c targets: General: < 7.0%; Elderly/complex: < 7.5-8.0%; Strict (young, low hypoglycemia risk): < 6.5%.
        Step 1: Lifestyle intervention + Metformin (first-line unless contraindicated).
        Step 2: Add second agent based on comorbidities:
        - ASCVD/high CV risk: GLP-1 receptor agonist (liraglutide, semaglutide) or SGLT-2 inhibitor (empagliflozin)
        - Heart failure: SGLT-2 inhibitor preferred (empagliflozin, dapagliflozin)
        - CKD: SGLT-2 inhibitor (dapagliflozin, empagliflozin) or GLP-1 agonist
        - Obesity/weight loss priority: GLP-1 agonist (semaglutide, liraglutide) or SGLT-2 inhibitor
        - Hypoglycemia avoidance: DPP-4 inhibitor (sitagliptin), GLP-1 agonist, or SGLT-2 inhibitor
        - Cost constraint: Sulfonylurea (glipizide) or basal insulin
        Insulin therapy: Initiate when HbA1c ≥10% or severe hyperglycemia; start basal insulin (glargine 10 units/night).
        Monitoring: HbA1c every 3 months until stable, then every 6 months; fasting glucose daily if on insulin.
        Preventive care: Annual eye exam (retinopathy), foot exam, urine albumin-creatinine ratio (nephropathy).
        Blood pressure target: < 130/80 mmHg; use ACE inhibitor or ARB.
        Lipid management: High-intensity statin for most patients ≥40 years with diabetes.
        """,
        'category': 'clinical_guideline'
    },
    {
        'title': 'Acute MI (STEMI) Management Protocol',
        'content': """
        STEMI management requires rapid reperfusion — time is muscle.
        Door-to-balloon time target: ≤90 minutes for primary PCI (preferred reperfusion strategy).
        Door-to-needle time for fibrinolysis: ≤30 minutes (if PCI unavailable within 120 minutes).
        Immediate medications (MONA mnemonic):
        - Morphine: 2-4 mg IV for refractory pain (use cautiously; may worsen outcomes)
        - Oxygen: Only if SpO2 < 90% (avoid routine supplemental O2)
        - Nitrates: SL nitroglycerin × 3 doses; avoid if RV infarct, hypotension, or phosphodiesterase inhibitor use
        - Aspirin: 325 mg chewed immediately; then 81 mg daily indefinitely
        Antiplatelet therapy: Dual antiplatelet therapy (DAPT) — aspirin + P2Y12 inhibitor (ticagrelor 180 mg load preferred over clopidogrel).
        Anticoagulation: Unfractionated heparin (UFH) or bivalirudin (preferred in high bleeding risk).
        Post-PCI medications: Beta-blocker (metoprolol), ACE inhibitor (lisinopril), high-intensity statin (atorvastatin 80 mg).
        Secondary prevention: DAPT for 12 months post-PCI (aspirin + ticagrelor/clopidogrel); reassess after 12 months.
        Cardiac rehabilitation: Refer all STEMI patients to cardiac rehab program.
        Complications: Monitor for cardiogenic shock, arrhythmias, mechanical complications (VSD, free wall rupture, papillary muscle rupture).
        """,
        'category': 'clinical_guideline'
    },
    # ── Symptom/Diagnosis ──
    {
        'title': 'Chest Pain Differential Diagnosis and Evaluation',
        'content': """
        Chest pain evaluation requires rapid risk stratification to identify life-threatening causes.
        Life-threatening causes to rule out first (the 5 Killers): ACS, aortic dissection, pulmonary embolism,
        tension pneumothorax, esophageal rupture.
        ACS features: Substernal pressure, radiation to jaw/arm/shoulder, diaphoresis, dyspnea, nausea.
        ECG changes: ST elevation (STEMI), ST depression/T-wave inversion (NSTEMI/UA), new LBBB.
        Troponin interpretation: Troponin I/T rise begins 2-4 hours after myocardial injury; peak at 12-24 hours.
        Use high-sensitivity troponin (hs-cTn) for rapid rule-in/rule-out (0h/1h or 0h/2h protocol).
        Aortic dissection: Tearing/ripping pain radiating to back, hypertensive history, pulse differential, mediastinal widening on CXR.
        PE features: Pleuritic chest pain, dyspnea, tachycardia; Wells score for pre-test probability; D-dimer if low probability.
        Pneumothorax: Sudden pleuritic pain, decreased breath sounds, tracheal deviation (tension).
        GERD/esophageal: Burning, acid taste, worse lying down, relieved by antacids.
        Musculoskeletal: Reproducible on palpation, positional, no radiation.
        Risk stratification tools: TIMI score, GRACE score, HEART score (for ED chest pain evaluation).
        HEART score ≥4: High risk (major adverse cardiac event risk > 13%); admit and monitor.
        """,
        'category': 'symptom_diagnosis'
    },
    {
        'title': 'Pneumonia Diagnosis and Management',
        'content': """
        Community-acquired pneumonia (CAP) diagnosis requires new pulmonary infiltrate on imaging plus ≥2 clinical features:
        fever/hypothermia, leukocytosis/leukopenia, purulent sputum, new cough, pleuritic chest pain, decreased breath sounds.
        Severity assessment: PSI (Pneumonia Severity Index) or CURB-65 score.
        CURB-65 criteria: Confusion, Urea > 7 mmol/L, RR ≥30/min, BP < 90 systolic or ≤ 60 diastolic, Age ≥65.
        CURB-65 score 0-1: Outpatient treatment; Score 2: Hospital admission; Score 3-5: ICU consideration.
        Outpatient CAP treatment: Amoxicillin 1g TID × 5-7 days; atypical coverage: azithromycin or doxycycline.
        Inpatient (non-ICU) CAP: Beta-lactam (ampicillin-sulbactam or ceftriaxone) + macrolide (azithromycin).
        ICU CAP: Antipseudomonal beta-lactam + azithromycin OR fluoroquinolone + antiPseudomonal coverage if risk factors.
        HCAP/HAP: Broader coverage: antipseudomonal beta-lactam + aminoglycoside or fluoroquinolone.
        Duration: CAP 5-7 days; HAP/VAP 7-8 days (de-escalate based on clinical response and cultures).
        Legionella/pneumococcal urinary antigen testing for hospitalized CAP.
        Follow-up chest X-ray: Not routinely needed at 6 weeks in young patients; needed in smokers ≥50 to rule out malignancy.
        Vaccination: Pneumococcal vaccine (PCV20 or PCV15+PPSV23) for all adults ≥65 and high-risk groups.
        """,
        'category': 'symptom_diagnosis'
    },
    {
        'title': 'Acute Kidney Injury (AKI) Diagnosis and Management',
        'content': """
        AKI definition (KDIGO): Increase in serum creatinine ≥0.3 mg/dL within 48 hours, OR ≥1.5× baseline within 7 days,
        OR urine output < 0.5 mL/kg/hr for ≥6 hours.
        KDIGO Staging: Stage 1: Cr 1.5-1.9× baseline or ≥0.3 mg/dL rise; UO < 0.5 mL/kg/hr for 6-12 hrs.
        Stage 2: Cr 2.0-2.9× baseline; UO < 0.5 mL/kg/hr for ≥12 hrs.
        Stage 3: Cr ≥3× baseline or ≥4.0 mg/dL or RRT initiated; UO < 0.3 mL/kg/hr for ≥24 hrs or anuria ≥12 hrs.
        Pre-renal causes: Volume depletion, cardiac failure, hepatorenal syndrome — FeNa < 1%, BUN/Cr ratio > 20.
        Intrinsic renal: ATN (ischemia, nephrotoxins), GN, interstitial nephritis — FeNa > 2%, granular casts on UA.
        Post-renal: Obstruction (BPH, stones) — bladder scan, renal ultrasound.
        Management: Identify and treat underlying cause, discontinue nephrotoxins (NSAIDs, aminoglycosides, contrast).
        Fluid management: Volume resuscitation for prerenal; avoid fluid overload in oliguric intrinsic AKI.
        Indications for emergent dialysis (AEIOU): Acidosis (pH < 7.1), Electrolyte abnormalities (K+ > 6.5),
        Ingestion (toxin), fluid Overload refractory to diuretics, Uremia (pericarditis, encephalopathy).
        Medications requiring dose adjustment in AKI: antibiotics, anticoagulants, analgesics — avoid NSAIDs.
        """,
        'category': 'symptom_diagnosis'
    },
    {
        'title': 'Stroke Recognition and Acute Management',
        'content': """
        Ischemic stroke accounts for 87% of all strokes. Hemorrhagic stroke: 13% (ICH: 10%, SAH: 3%).
        FAST recognition: Face drooping, Arm weakness, Speech difficulty, Time to call 911.
        BE-FAST adds: Balance problems, Eyes (vision changes).
        IV tPA (alteplase): Indicated for ischemic stroke within 4.5 hours of symptom onset.
        tPA dose: 0.9 mg/kg (maximum 90 mg); 10% bolus over 1 minute, remainder over 60 minutes.
        tPA contraindications: BP > 185/110 (treat first), recent surgery, anticoagulation with INR > 1.7, platelets < 100,000.
        Mechanical thrombectomy: For large vessel occlusion (LVO) up to 24 hours in selected patients (DAWN/DEFUSE-3 criteria).
        Blood pressure management: Before tPA: treat if BP > 185/110 (use labetalol/nicardipine).
        After tPA: maintain BP < 180/105 for 24 hours.
        Hemorrhagic stroke: Reverse anticoagulation (PCC for warfarin, andexanet alfa for Factor Xa inhibitors);
        neurosurgical consultation for hematoma > 30 mL or cerebellar hemorrhage.
        Secondary prevention (ischemic): Antiplatelet therapy (aspirin ± dipyridamole, or clopidogrel).
        For cardioembolic (Afib): anticoagulation (DOAC preferred over warfarin).
        """,
        'category': 'symptom_diagnosis'
    },
    # ── Treatment Protocols ──
    {
        'title': 'COPD Exacerbation Treatment Protocol',
        'content': """
        COPD exacerbation: Acute worsening of respiratory symptoms beyond normal day-to-day variation requiring additional treatment.
        Severity assessment: Mild (treated at home), Moderate (ED visit/short hospitalization), Severe (ICU/ventilation).
        Bronchodilator therapy: Albuterol (SABA) 2.5-5 mg via nebulizer every 20 minutes × 3 then every 1-4 hours.
        Ipratropium (SAMA) 0.5 mg via nebulizer every 4-6 hours; can combine with albuterol (DuoNeb).
        Systemic corticosteroids: Prednisone 40 mg orally × 5 days (equivalent to 7-14 day course per REDUCE trial).
        Antibiotics: Indicated if purulent sputum + increased dyspnea OR increased sputum volume.
        Antibiotic choice: Amoxicillin-clavulanate 875/125 mg twice daily × 5-7 days; or azithromycin × 5 days;
        severe/Pseudomonas risk: fluoroquinolone (levofloxacin 750 mg daily).
        Oxygen therapy: Target SpO2 88-92% (avoid over-oxygenation; risk of hypercapnic respiratory failure).
        NIV (BiPAP/CPAP): Indicated for acute hypercapnic respiratory failure (PaCO2 > 45 with pH < 7.35).
        NIV reduces need for intubation, mortality, and length of stay.
        Intubation indications: Failure of NIV, severe acidosis (pH < 7.25), deteriorating mental status.
        Discharge criteria: Stable on oral medications, SpO2 ≥92% on ≤ 2L NC, adequate inhaler technique.
        Follow-up: Within 2-4 weeks; reassess inhaler technique, optimize maintenance therapy (LABA+LAMA+ICS if GOLD 3-4).
        """,
        'category': 'treatment_protocol'
    },
    {
        'title': 'Anticoagulation for Atrial Fibrillation',
        'content': """
        Atrial fibrillation stroke risk assessment: Use CHA₂DS₂-VASc score.
        CHA₂DS₂-VASc: CHF(1), Hypertension(1), Age ≥75(2), Diabetes(1), Stroke/TIA(2), Vascular disease(1),
        Age 65-74(1), Sex (Female)(1).
        Anticoagulation thresholds: Males: score ≥2; Females: score ≥3 (warfarin/DOAC recommended).
        Score 1 (males) or 2 (females): Consider anticoagulation; assess bleeding risk (HAS-BLED score).
        DOAC options (preferred over warfarin for non-valvular AF):
        - Apixaban: 5 mg twice daily (2.5 mg twice daily if ≥2 of: age ≥80, weight ≤60 kg, Cr ≥1.5 mg/dL)
        - Rivaroxaban: 20 mg once daily with evening meal
        - Dabigatran: 150 mg twice daily (110 mg twice daily for age ≥80 or high bleeding risk)
        - Edoxaban: 60 mg once daily (30 mg if CrCl 15-50, weight ≤60 kg, or P-gp inhibitors)
        Warfarin: Use for mechanical valves or significant mitral stenosis (DOACs not approved).
        Rate control: Target resting HR < 110 bpm; use beta-blockers, rate-limiting CCBs (diltiazem/verapamil), or digoxin.
        Cardioversion: Anticoagulate for ≥3 weeks before and ≥4 weeks after cardioversion; or TEE to rule out thrombus.
        Bleeding management: Reversal agents — Idarucizumab for dabigatran; Andexanet alfa for Factor Xa inhibitors.
        """,
        'category': 'treatment_protocol'
    },
    {
        'title': 'Acute Pain Management Protocol',
        'content': """
        Multimodal analgesia is the preferred approach to minimize opioid use while providing effective pain control.
        WHO analgesic ladder: Step 1 (mild pain): non-opioids (acetaminophen, NSAIDs); Step 2 (moderate): weak opioids;
        Step 3 (severe): strong opioids.
        Acetaminophen: 325-1000 mg every 4-6 hours; maximum 4000 mg/day (3000 mg/day in elderly or liver disease).
        IV acetaminophen: 1000 mg every 6 hours; use in NPO patients or for opioid-sparing effect.
        NSAIDs: Ibuprofen 400-800 mg every 6-8 hours with food; avoid in GI disease, renal impairment, CV risk.
        Ketorolac IV: 15-30 mg IV/IM; limit to 5 days; effective for moderate-severe pain.
        Opioid analgesics: Morphine 2-4 mg IV every 3-4 hours; hydromorphone 0.4-1 mg IV every 3-4 hours (6× more potent).
        Opioid conversion: Morphine 10 mg IV = oxycodone 15 mg PO = hydrocodone 20 mg PO.
        Adjuvant analgesics: Gabapentin/pregabalin for neuropathic pain; ketamine 0.1-0.3 mg/kg/hr infusion (opioid-sparing);
        regional anesthesia (nerve blocks) for post-surgical/trauma pain.
        Opioid risk mitigation: PDMP check, naloxone co-prescription, avoid concurrent benzodiazepines.
        Opioid-induced constipation: Prophylactic senna + docusate; consider methylnaltrexone for severe cases.
        """,
        'category': 'treatment_protocol'
    },
    {
        'title': 'Deep Vein Thrombosis (DVT) Treatment Protocol',
        'content': """
        DVT diagnosis: Wells score for pre-test probability + D-dimer + duplex ultrasound.
        Wells DVT score: Active cancer(1), Paralysis/recent immobilization(1), Bedridden ≥3 days/surgery in 4 weeks(1),
        localized tenderness(1), entire leg swollen(1), calf swelling >3cm(1), pitting edema(1), collateral veins(1).
        Wells ≥2: High probability; proceed directly to ultrasound.
        Wells 0-1: Low probability; D-dimer first; if negative, DVT excluded.
        Treatment — first episode proximal DVT/PE: Anticoagulate for minimum 3 months.
        DOACs preferred: Rivaroxaban 15 mg twice daily × 21 days, then 20 mg daily; OR apixaban 10 mg twice daily × 7 days, then 5 mg twice daily.
        Alternative: LMWH (enoxaparin 1 mg/kg twice daily) bridge to warfarin (target INR 2-3).
        Provoked DVT (transient risk factor): 3-month treatment sufficient.
        Unprovoked DVT: Consider extended therapy (reassess risk/benefit at 3 months).
        Cancer-associated DVT: LMWH or rivaroxaban/edoxaban preferred; indefinite treatment while cancer active.
        Massive PE with hemodynamic instability: Consider systemic thrombolysis (alteplase 100 mg over 2 hours) or catheter-directed therapy.
        Inferior vena cava (IVC) filter: Reserved for patients with PE/DVT and absolute contraindication to anticoagulation.
        Compression stockings: Not recommended routinely for prevention of post-thrombotic syndrome.
        """,
        'category': 'treatment_protocol'
    }
]

print(f'✅ Loaded {len(MEDICAL_CORPUS)} medical documents')
for doc in MEDICAL_CORPUS:
    print(f"  [{doc['category']}] {doc['title']}")

In [ ]:
def ingest_documents(corpus: List[Dict]) -> List[Document]:
    """
    Convert raw corpus entries into LangChain Documents.
    
    Args:
        corpus: List of dicts with 'title', 'content', 'category'
    Returns:
        List of LangChain Document objects with metadata
    """
    docs = []
    for item in corpus:
        doc = Document(
            page_content=item['content'].strip(),
            metadata={
                'title': item['title'],
                'category': item['category'],
                'source': f"internal_db/{item['category']}/{item['title'].lower().replace(' ', '_')}",
                'doc_id': str(uuid.uuid4())
            }
        )
        docs.append(doc)
    return docs


def chunk_documents(docs: List[Document], chunk_size: int = 500, chunk_overlap: int = 50) -> List[Document]:
    """
    Split documents into chunks using RecursiveCharacterTextSplitter.
    
    Args:
        docs: List of LangChain Documents
        chunk_size: Maximum characters per chunk (default 500)
        chunk_overlap: Overlap between consecutive chunks (default 50)
    Returns:
        List of chunked Document objects
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', ''],
        length_function=len
    )
    chunks = splitter.split_documents(docs)
    # Preserve and annotate metadata
    for i, chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = i
        chunk.metadata['chunk_length'] = len(chunk.page_content)
    return chunks


# Ingest and chunk
raw_docs = ingest_documents(MEDICAL_CORPUS)
chunked_docs = chunk_documents(raw_docs, CHUNK_SIZE, CHUNK_OVERLAP)

print(f'✅ Raw documents: {len(raw_docs)}')
print(f'✅ Chunked documents: {len(chunked_docs)}')
print(f'\nSample chunk:\n{"-"*50}')
print(chunked_docs[0].page_content[:300])
print(f'Metadata: {chunked_docs[0].metadata}')

## 🗄️ Section 3: ChromaDB Vector Store Setup & Indexing

In [ ]:
# Initialize Sentence Transformer embeddings
print('Loading sentence transformer model...')
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)
print('✅ Embedding model loaded: all-MiniLM-L6-v2')

# Test embedding
test_emb = embedding_model.embed_query('test medical query')
print(f'✅ Embedding dimensions: {len(test_emb)}')

In [ ]:
def build_vector_store(chunks: List[Document], persist_dir: str, collection: str) -> Chroma:
    """
    Build and persist ChromaDB vector store from document chunks.
    Clears existing collection to ensure fresh indexing.
    
    Args:
        chunks: List of chunked Documents
        persist_dir: Local directory for ChromaDB persistence
        collection: ChromaDB collection name
    Returns:
        Chroma vector store instance
    """
    import shutil
    # Clear old DB for clean build
    if Path(persist_dir).exists():
        shutil.rmtree(persist_dir)
        print(f'  Cleared existing ChromaDB at {persist_dir}')
    
    print(f'  Indexing {len(chunks)} chunks...')
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir,
        collection_name=collection
    )
    print(f'✅ ChromaDB built with {vectorstore._collection.count()} embeddings')
    return vectorstore


vectorstore = build_vector_store(chunked_docs, CHROMA_PERSIST_DIR, COLLECTION_NAME)
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': RETRIEVAL_K}
)

# Verify retrieval
test_query = 'What is the maximum daily dose of metformin?'
test_results = retriever.get_relevant_documents(test_query)
print(f'\n✅ Test retrieval for: "{test_query}"')
for i, doc in enumerate(test_results[:2]):
    print(f'  [{i+1}] {doc.metadata["title"]} — {doc.page_content[:100]}...')

## 🧠 Section 4: LangGraph CRAG Pipeline Implementation

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Initialize Gemini LLM
# ─────────────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model='gemini-1.5-flash',          # Use gemini-1.5-pro for higher quality
    temperature=0.1,                    # Low temp for factual medical QA
    max_output_tokens=2048,
    google_api_key=GOOGLE_API_KEY
)

# Tavily web search (fallback)
web_search_tool = TavilySearchResults(
    max_results=3,
    search_depth='advanced',
    include_answer=True,
    include_raw_content=False
)

print('✅ Gemini LLM initialized')
print('✅ Tavily web search initialized')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# LANGGRAPH STATE DEFINITION
# ─────────────────────────────────────────────────────────────────────
class CRAGState(TypedDict):
    """State passed between nodes in the CRAG LangGraph pipeline."""
    # Input
    question: str
    # Retrieved documents
    documents: List[Document]
    # Graded documents with relevance labels
    graded_documents: List[Dict[str, Any]]
    # Routing decision
    route_decision: str  # 'generate' | 'transform' | 'web_search'
    # Transformed query (if applicable)
    transformed_query: Optional[str]
    # Web search results (if applicable)
    web_results: Optional[List[Dict]]
    # Generated answer
    generation: str
    # Hallucination check result
    hallucination_check: str  # 'grounded' | 'not_grounded'
    # Number of retries (prevent infinite loops)
    retry_count: int
    # Source citations
    sources: List[str]
    # Pipeline path taken (for observability)
    pipeline_path: List[str]

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# NODE 1: RETRIEVE
# ─────────────────────────────────────────────────────────────────────
def retrieve_node(state: CRAGState) -> CRAGState:
    """
    Performs k-NN similarity search against ChromaDB.
    Uses the current question (or transformed query if available).
    """
    query = state.get('transformed_query') or state['question']
    print(f'\n[RETRIEVE] Query: {query[:80]}...' if len(query) > 80 else f'\n[RETRIEVE] Query: {query}')
    
    docs = retriever.get_relevant_documents(query)
    
    path = state.get('pipeline_path', []) + ['RETRIEVE']
    print(f'  → Retrieved {len(docs)} documents')
    
    return {
        **state,
        'documents': docs,
        'pipeline_path': path
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 2: GRADE DOCUMENTS
# ─────────────────────────────────────────────────────────────────────
GRADE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """
You are a medical information relevance grader.
Given a clinical question and a retrieved document chunk, assess whether
the document contains information that would help answer the question.

Respond ONLY with a JSON object:
{{
  "relevance": "RELEVANT" | "IRRELEVANT" | "AMBIGUOUS",
  "reason": "<brief one-line reason>"
}}

Rules:
- RELEVANT: Document directly addresses the question (dosage, mechanism, diagnosis, protocol)
- IRRELEVANT: Document has no useful overlap with the question
- AMBIGUOUS: Document partially overlaps or contains tangential information
"""),
    ('human', 'Clinical Question: {question}\n\nDocument:\n{document}')
])

grade_chain = GRADE_PROMPT | llm | StrOutputParser()


def grade_documents_node(state: CRAGState) -> CRAGState:
    """
    LLM grades each retrieved document as RELEVANT / IRRELEVANT / AMBIGUOUS.
    Filters and labels documents for routing decision.
    """
    print('\n[GRADE DOCUMENTS]')
    question = state['question']
    documents = state['documents']
    
    graded = []
    for i, doc in enumerate(documents):
        try:
            result = grade_chain.invoke({
                'question': question,
                'document': doc.page_content[:600]  # Truncate to save tokens
            })
            # Parse JSON from response (handle potential markdown wrapping)
            result_clean = result.strip().replace('```json', '').replace('```', '').strip()
            grade_data = json.loads(result_clean)
            relevance = grade_data.get('relevance', 'AMBIGUOUS')
        except Exception as e:
            # Default to AMBIGUOUS on parse failure
            relevance = 'AMBIGUOUS'
            grade_data = {'relevance': 'AMBIGUOUS', 'reason': str(e)[:50]}
        
        graded.append({
            'document': doc,
            'relevance': relevance,
            'reason': grade_data.get('reason', ''),
            'title': doc.metadata.get('title', 'Unknown')
        })
        print(f'  [{i+1}] {doc.metadata.get("title", "Doc")[:40]} → {relevance}')
    
    path = state.get('pipeline_path', []) + ['GRADE_DOCUMENTS']
    return {
        **state,
        'graded_documents': graded,
        'pipeline_path': path
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 3: ROUTE (conditional edges)
# ─────────────────────────────────────────────────────────────────────
def route_documents(state: CRAGState) -> str:
    """
    Routing function for conditional edges in LangGraph.
    
    Rules:
    - All RELEVANT → 'generate'
    - Some IRRELEVANT/AMBIGUOUS → 'transform_query'
    - All IRRELEVANT/AMBIGUOUS → 'web_search'
    - Max 2 retries to prevent infinite loops
    """
    graded = state.get('graded_documents', [])
    retry_count = state.get('retry_count', 0)
    
    relevant_count = sum(1 for g in graded if g['relevance'] == 'RELEVANT')
    irrelevant_count = sum(1 for g in graded if g['relevance'] == 'IRRELEVANT')
    total = len(graded)
    
    print(f'\n[ROUTE] Relevant: {relevant_count}/{total} | Retry: {retry_count}')
    
    # Prevent infinite loops
    if retry_count >= 2:
        print('  → Max retries reached → generate (best effort)')
        return 'generate'
    
    if relevant_count == total:          # All relevant
        print('  → All relevant → generate')
        return 'generate'
    elif irrelevant_count == total:      # All irrelevant
        print('  → All irrelevant → web_search')
        return 'web_search'
    else:                                # Mixed
        print('  → Mixed relevance → transform_query')
        return 'transform_query'


# ─────────────────────────────────────────────────────────────────────
# NODE 4: TRANSFORM QUERY
# ─────────────────────────────────────────────────────────────────────
TRANSFORM_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """
You are a medical query optimization specialist.
Rewrite the given clinical question to improve retrieval from a medical knowledge base.
The rewritten query should:
1. Include relevant medical terminology (ICD codes, drug generic names, clinical acronyms)
2. Be more specific and precise
3. Expand abbreviations
4. Focus on the core clinical concept
Respond with ONLY the rewritten query, no explanation.
"""),
    ('human', 'Original query: {question}\nRewritten query:')
])

transform_chain = TRANSFORM_PROMPT | llm | StrOutputParser()


def transform_query_node(state: CRAGState) -> CRAGState:
    """
    Rewrites the query using medical terminology for improved retrieval.
    Increments retry_count to prevent infinite loops.
    """
    print('\n[TRANSFORM QUERY]')
    original = state['question']
    transformed = transform_chain.invoke({'question': original}).strip()
    print(f'  Original:    {original}')
    print(f'  Transformed: {transformed}')
    
    path = state.get('pipeline_path', []) + ['TRANSFORM_QUERY']
    return {
        **state,
        'transformed_query': transformed,
        'retry_count': state.get('retry_count', 0) + 1,
        'pipeline_path': path
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 5: WEB SEARCH FALLBACK
# ─────────────────────────────────────────────────────────────────────
def web_search_node(state: CRAGState) -> CRAGState:
    """
    Fallback web search using Tavily API when all retrieved docs are irrelevant.
    Converts search results into Document objects for uniform downstream processing.
    """
    print('\n[WEB SEARCH FALLBACK]')
    query = state.get('transformed_query') or state['question']
    
    try:
        results = web_search_tool.invoke({'query': f'medical {query}'})
        # Convert Tavily results to Document objects
        web_docs = []
        for r in results:
            if isinstance(r, dict):
                web_docs.append(Document(
                    page_content=r.get('content', r.get('snippet', '')),
                    metadata={
                        'title': r.get('title', 'Web Result'),
                        'source': r.get('url', 'web'),
                        'category': 'web_search'
                    }
                ))
        
        print(f'  → Found {len(web_docs)} web results')
        path = state.get('pipeline_path', []) + ['WEB_SEARCH']
        return {
            **state,
            'documents': web_docs,
            'web_results': results,
            'pipeline_path': path
        }
    except Exception as e:
        print(f'  ⚠️  Web search failed: {e}. Using original documents.')
        path = state.get('pipeline_path', []) + ['WEB_SEARCH_FAILED']
        return {**state, 'pipeline_path': path}


# ─────────────────────────────────────────────────────────────────────
# NODE 6: GENERATE
# ─────────────────────────────────────────────────────────────────────
GENERATE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """
You are an expert medical AI assistant helping clinicians with evidence-based answers.
Answer the clinical question using ONLY the provided context documents.

RULES:
1. Base your answer strictly on the provided context — do not introduce external knowledge
2. Be precise and clinically accurate
3. Include specific values (dosages, thresholds, timeframes) from the context
4. If context is insufficient, clearly state what information is missing
5. End with: SOURCES: [list the document titles used]

Format: Clinical Answer → Sources
"""),
    ('human', """
Clinical Question: {question}

Context Documents:
{context}

Provide a concise, clinically actionable answer with source citations.
""")
])

generate_chain = GENERATE_PROMPT | llm | StrOutputParser()


def generate_node(state: CRAGState) -> CRAGState:
    """
    Generates grounded answer using Gemini LLM with relevant context.
    Uses only RELEVANT and AMBIGUOUS graded documents (filters IRRELEVANT).
    """
    print('\n[GENERATE]')
    question = state['question']
    graded_docs = state.get('graded_documents', [])
    all_docs = state.get('documents', [])
    
    # Use relevant+ambiguous graded docs; fall back to all docs if grading absent
    if graded_docs:
        context_docs = [g['document'] for g in graded_docs
                       if g['relevance'] in ('RELEVANT', 'AMBIGUOUS')]
        if not context_docs:
            context_docs = [g['document'] for g in graded_docs]  # Use all if none pass filter
    else:
        context_docs = all_docs
    
    # Format context with titles
    context_str = '\n\n---\n'.join([
        f"[SOURCE: {doc.metadata.get('title', 'Unknown')}]\n{doc.page_content}"
        for doc in context_docs[:5]  # Limit to top 5 for context window
    ])
    
    generation = generate_chain.invoke({
        'question': question,
        'context': context_str
    })
    
    # Extract sources from generation
    sources = [doc.metadata.get('title', 'Unknown') for doc in context_docs]
    
    print(f'  → Generated answer ({len(generation)} chars)')
    path = state.get('pipeline_path', []) + ['GENERATE']
    return {
        **state,
        'generation': generation,
        'sources': sources,
        'pipeline_path': path
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 7: HALLUCINATION CHECK
# ─────────────────────────────────────────────────────────────────────
HALLUCINATION_CHECK_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """
You are a medical fact-checking AI verifying answer groundedness.
Given a generated answer and the source context, determine whether every factual
claim in the answer is supported by the provided context.

Respond ONLY with a JSON object:
{{
  "verdict": "grounded" | "not_grounded",
  "ungrounded_claims": ["claim1", "claim2"],
  "confidence": 0.0-1.0,
  "reason": "brief explanation"
}}

Use 'not_grounded' only if specific factual claims (dosages, diagnoses, thresholds)
cannot be verified in the context. Extrapolations and inferences count as grounded.
"""),
    ('human', """
Generated Answer:
{generation}

Source Context:
{context}
""")
])

hallucination_chain = HALLUCINATION_CHECK_PROMPT | llm | StrOutputParser()


def hallucination_check_node(state: CRAGState) -> CRAGState:
    """
    Verifies generated answer is grounded in retrieved context.
    Acts as a safety gate before returning to user.
    """
    print('\n[HALLUCINATION CHECK]')
    generation = state['generation']
    docs = state.get('documents', [])
    
    context_str = '\n'.join([doc.page_content[:400] for doc in docs[:3]])
    
    try:
        result = hallucination_chain.invoke({
            'generation': generation[:1500],
            'context': context_str[:2000]
        })
        result_clean = result.strip().replace('```json', '').replace('```', '').strip()
        check_data = json.loads(result_clean)
        verdict = check_data.get('verdict', 'grounded')
        confidence = check_data.get('confidence', 1.0)
    except Exception as e:
        verdict = 'grounded'  # Default to grounded on parse failure
        confidence = 0.7
        print(f'  ⚠️  Parse error: {e}')
    
    print(f'  → Verdict: {verdict} (confidence: {confidence:.2f})')
    path = state.get('pipeline_path', []) + ['HALLUCINATION_CHECK']
    return {
        **state,
        'hallucination_check': verdict,
        'pipeline_path': path
    }


def should_regenerate(state: CRAGState) -> str:
    """Conditional edge after hallucination check."""
    if state.get('hallucination_check') == 'not_grounded' and state.get('retry_count', 0) < 2:
        print('  → Not grounded → regenerating with web search')
        return 'web_search'
    return 'end'


print('✅ All CRAG nodes defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# BUILD LANGGRAPH PIPELINE
# ─────────────────────────────────────────────────────────────────────
def build_crag_graph() -> CompiledGraph:
    """
    Assembles the full CRAG LangGraph pipeline with:
    - Stateful nodes
    - Conditional routing edges
    - Retry loops with circuit breaker
    """
    graph = StateGraph(CRAGState)
    
    # ── Add Nodes ──
    graph.add_node('retrieve', retrieve_node)
    graph.add_node('grade_documents', grade_documents_node)
    graph.add_node('transform_query', transform_query_node)
    graph.add_node('web_search', web_search_node)
    graph.add_node('generate', generate_node)
    graph.add_node('hallucination_check', hallucination_check_node)
    
    # ── Entry Point ──
    graph.set_entry_point('retrieve')
    
    # ── Static Edges ──
    graph.add_edge('retrieve', 'grade_documents')
    graph.add_edge('transform_query', 'retrieve')   # Re-retrieve with better query
    graph.add_edge('web_search', 'generate')         # Web results → generate directly
    graph.add_edge('generate', 'hallucination_check')
    
    # ── Conditional Edge: Route after grading ──
    graph.add_conditional_edges(
        'grade_documents',
        route_documents,
        {
            'generate': 'generate',
            'transform_query': 'transform_query',
            'web_search': 'web_search'
        }
    )
    
    # ── Conditional Edge: After hallucination check ──
    graph.add_conditional_edges(
        'hallucination_check',
        should_regenerate,
        {
            'web_search': 'web_search',
            'end': END
        }
    )
    
    return graph.compile()


crag_pipeline = build_crag_graph()
print('✅ CRAG LangGraph pipeline compiled')

# Visualize the graph
try:
    from IPython.display import Image, display
    graph_png = crag_pipeline.get_graph().draw_mermaid_png()
    display(Image(graph_png))
    print('✅ Graph visualization displayed')
except Exception as e:
    print(f'Graph visualization skipped: {e}')
    # Print Mermaid diagram as fallback
    print('\nMermaid Graph:')
    print(crag_pipeline.get_graph().draw_mermaid())

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CRAG PIPELINE RUNNER
# ─────────────────────────────────────────────────────────────────────
def run_crag(question: str, verbose: bool = True) -> Dict[str, Any]:
    """
    Run a question through the full CRAG pipeline.
    
    Args:
        question: Clinical question string
        verbose: Print pipeline trace
    Returns:
        Dict with answer, sources, pipeline_path, hallucination_check
    """
    initial_state: CRAGState = {
        'question': question,
        'documents': [],
        'graded_documents': [],
        'route_decision': '',
        'transformed_query': None,
        'web_results': None,
        'generation': '',
        'hallucination_check': '',
        'retry_count': 0,
        'sources': [],
        'pipeline_path': []
    }
    
    if verbose:
        print(f'\n{"="*60}')
        print(f'CRAG QUERY: {question}')
        print('='*60)
    
    final_state = crag_pipeline.invoke(initial_state)
    
    if verbose:
        print(f'\n{"─"*60}')
        print(f'PIPELINE PATH: {" → ".join(final_state["pipeline_path"])}')
        print(f'HALLUCINATION CHECK: {final_state["hallucination_check"]}')
        print(f'SOURCES: {final_state["sources"]}')
        print(f'\nANSWER:\n{final_state["generation"]}')
    
    return {
        'question': question,
        'answer': final_state['generation'],
        'sources': final_state['sources'],
        'pipeline_path': final_state['pipeline_path'],
        'hallucination_check': final_state['hallucination_check'],
        'documents': final_state['documents'],
        'graded_documents': final_state.get('graded_documents', [])
    }


# Test the CRAG pipeline
test_result = run_crag('What is the maximum daily dose of metformin and what monitoring is required?')

## 📊 Section 5: Evaluation Dataset Creation

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# SYNTHETIC EVALUATION DATASET — 50 QA PAIRS
# Covers: drug dosage, clinical guidelines, symptom/diagnosis, treatment
# ─────────────────────────────────────────────────────────────────────
EVAL_DATASET = [
    # ── Drug Dosage (15 questions) ──
    {
        'id': 1, 'category': 'drug_dosage',
        'question': 'What is the maximum daily dose of metformin?',
        'expected_answer': '2550 mg per day',
        'relevant_doc_ids': ['Metformin Clinical Reference'],
        'keywords': ['2550', 'maximum', 'metformin']
    },
    {
        'id': 2, 'category': 'drug_dosage',
        'question': 'What is the initial dose of lisinopril for hypertension?',
        'expected_answer': '10 mg once daily',
        'relevant_doc_ids': ['Lisinopril ACE Inhibitor Reference'],
        'keywords': ['10 mg', 'lisinopril', 'once daily']
    },
    {
        'id': 3, 'category': 'drug_dosage',
        'question': 'What is the target INR range for atrial fibrillation on warfarin?',
        'expected_answer': '2.0 to 3.0',
        'relevant_doc_ids': ['Warfarin Anticoagulation Protocol'],
        'keywords': ['2.0', '3.0', 'INR', 'atrial fibrillation']
    },
    {
        'id': 4, 'category': 'drug_dosage',
        'question': 'What is the standard adult dose of amoxicillin for common infections?',
        'expected_answer': '500 mg every 8 hours or 875 mg every 12 hours',
        'relevant_doc_ids': ['Amoxicillin Antibiotic Reference'],
        'keywords': ['500 mg', '875 mg', 'amoxicillin']
    },
    {
        'id': 5, 'category': 'drug_dosage',
        'question': 'What dose of atorvastatin is considered high-intensity statin therapy?',
        'expected_answer': '40-80 mg daily',
        'relevant_doc_ids': ['Atorvastatin Lipid Management Reference'],
        'keywords': ['40', '80 mg', 'high-intensity']
    },
    {
        'id': 6, 'category': 'drug_dosage',
        'question': 'What is the metformin contraindication threshold for eGFR?',
        'expected_answer': 'eGFR less than 30 mL/min/1.73m²',
        'relevant_doc_ids': ['Metformin Clinical Reference'],
        'keywords': ['30', 'eGFR', 'contraindication']
    },
    {
        'id': 7, 'category': 'drug_dosage',
        'question': 'What is the IV tPA dose for acute ischemic stroke?',
        'expected_answer': '0.9 mg/kg (maximum 90 mg)',
        'relevant_doc_ids': ['Stroke Recognition and Acute Management'],
        'keywords': ['0.9 mg/kg', '90 mg', 'alteplase']
    },
    {
        'id': 8, 'category': 'drug_dosage',
        'question': 'What is the standard dose of prednisone for COPD exacerbation?',
        'expected_answer': '40 mg orally for 5 days',
        'relevant_doc_ids': ['COPD Exacerbation Treatment Protocol'],
        'keywords': ['40 mg', 'prednisone', '5 days']
    },
    {
        'id': 9, 'category': 'drug_dosage',
        'question': 'What is the standard dose of apixaban for non-valvular atrial fibrillation?',
        'expected_answer': '5 mg twice daily',
        'relevant_doc_ids': ['Anticoagulation for Atrial Fibrillation'],
        'keywords': ['5 mg', 'apixaban', 'twice daily']
    },
    {
        'id': 10, 'category': 'drug_dosage',
        'question': 'What is the maximum daily dose of acetaminophen?',
        'expected_answer': '4000 mg per day (3000 mg in elderly)',
        'relevant_doc_ids': ['Acute Pain Management Protocol'],
        'keywords': ['4000 mg', 'acetaminophen', '3000']
    },
    {
        'id': 11, 'category': 'drug_dosage',
        'question': 'What are the DOAC dose reduction criteria for apixaban?',
        'expected_answer': '2.5 mg twice daily if 2 of: age ≥80, weight ≤60 kg, Cr ≥1.5 mg/dL',
        'relevant_doc_ids': ['Anticoagulation for Atrial Fibrillation'],
        'keywords': ['2.5 mg', 'reduction', 'apixaban']
    },
    {
        'id': 12, 'category': 'drug_dosage',
        'question': 'What is the loading dose of ticagrelor for acute MI?',
        'expected_answer': '180 mg loading dose',
        'relevant_doc_ids': ['Acute MI (STEMI) Management Protocol'],
        'keywords': ['180 mg', 'ticagrelor', 'load']
    },
    {
        'id': 13, 'category': 'drug_dosage',
        'question': 'What is the initial vasopressin dose in septic shock management?',
        'expected_answer': '0.03 units per minute',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['0.03', 'vasopressin', 'units/min']
    },
    {
        'id': 14, 'category': 'drug_dosage',
        'question': 'What is the enoxaparin dose for DVT treatment?',
        'expected_answer': '1 mg/kg twice daily subcutaneously',
        'relevant_doc_ids': ['Deep Vein Thrombosis (DVT) Treatment Protocol'],
        'keywords': ['1 mg/kg', 'enoxaparin', 'twice daily']
    },
    {
        'id': 15, 'category': 'drug_dosage',
        'question': 'What is the tidal volume target for ARDS ventilation in sepsis?',
        'expected_answer': '6 mL/kg ideal body weight',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['6 mL/kg', 'tidal volume', 'ARDS']
    },
    # ── Clinical Guidelines (12 questions) ──
    {
        'id': 16, 'category': 'clinical_guideline',
        'question': 'What are the components of the sepsis Hour-1 Bundle?',
        'expected_answer': 'Measure lactate, obtain blood cultures, administer antibiotics, 30 mL/kg crystalloid, vasopressors',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['lactate', 'blood cultures', 'antibiotics', '30 mL/kg']
    },
    {
        'id': 17, 'category': 'clinical_guideline',
        'question': 'What is the blood pressure classification for Stage 2 hypertension per ACC/AHA 2017?',
        'expected_answer': 'Blood pressure ≥140/≥90 mmHg',
        'relevant_doc_ids': ['Hypertension Management Guidelines (JNC 8 / ACC/AHA 2017)'],
        'keywords': ['140', '90', 'Stage 2', 'hypertension']
    },
    {
        'id': 18, 'category': 'clinical_guideline',
        'question': 'What is the HbA1c target for most adults with type 2 diabetes?',
        'expected_answer': 'Less than 7.0%',
        'relevant_doc_ids': ['Type 2 Diabetes Management Protocol'],
        'keywords': ['7.0%', 'HbA1c', 'target']
    },
    {
        'id': 19, 'category': 'clinical_guideline',
        'question': 'What is the door-to-balloon time target for STEMI?',
        'expected_answer': '90 minutes or less',
        'relevant_doc_ids': ['Acute MI (STEMI) Management Protocol'],
        'keywords': ['90 minutes', 'door-to-balloon', 'STEMI']
    },
    {
        'id': 20, 'category': 'clinical_guideline',
        'question': 'What is the blood glucose target for sepsis patients?',
        'expected_answer': '140-180 mg/dL',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['140', '180', 'glucose', 'insulin']
    },
    {
        'id': 21, 'category': 'clinical_guideline',
        'question': 'Which antihypertensive drug class is preferred for patients with CKD and proteinuria?',
        'expected_answer': 'ACE inhibitor or ARB',
        'relevant_doc_ids': ['Hypertension Management Guidelines (JNC 8 / ACC/AHA 2017)'],
        'keywords': ['ACE inhibitor', 'ARB', 'CKD', 'proteinuria']
    },
    {
        'id': 22, 'category': 'clinical_guideline',
        'question': 'Which SGLT-2 inhibitor is preferred for patients with heart failure and diabetes?',
        'expected_answer': 'Empagliflozin or dapagliflozin',
        'relevant_doc_ids': ['Type 2 Diabetes Management Protocol'],
        'keywords': ['empagliflozin', 'dapagliflozin', 'heart failure']
    },
    {
        'id': 23, 'category': 'clinical_guideline',
        'question': 'What is the minimum anticoagulation duration for a first proximal DVT episode?',
        'expected_answer': '3 months',
        'relevant_doc_ids': ['Deep Vein Thrombosis (DVT) Treatment Protocol'],
        'keywords': ['3 months', 'minimum', 'DVT']
    },
    {
        'id': 24, 'category': 'clinical_guideline',
        'question': 'What oxygen saturation target should be maintained in COPD exacerbation?',
        'expected_answer': 'SpO2 88-92%',
        'relevant_doc_ids': ['COPD Exacerbation Treatment Protocol'],
        'keywords': ['88', '92', 'SpO2', 'COPD']
    },
    {
        'id': 25, 'category': 'clinical_guideline',
        'question': 'What MAP target should be maintained in septic shock?',
        'expected_answer': 'Mean arterial pressure ≥65 mmHg',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['65 mmHg', 'MAP', 'septic shock']
    },
    {
        'id': 26, 'category': 'clinical_guideline',
        'question': 'What is the CHA2DS2-VASc score threshold for anticoagulation in males with AF?',
        'expected_answer': 'Score 2 or higher',
        'relevant_doc_ids': ['Anticoagulation for Atrial Fibrillation'],
        'keywords': ['2', 'CHA2DS2-VASc', 'male', 'anticoagulation']
    },
    {
        'id': 27, 'category': 'clinical_guideline',
        'question': 'What is the CURB-65 score threshold for hospital admission in pneumonia?',
        'expected_answer': 'Score 2 warrants hospital admission',
        'relevant_doc_ids': ['Pneumonia Diagnosis and Management'],
        'keywords': ['2', 'CURB-65', 'hospital', 'admission']
    },
    # ── Symptom/Diagnosis (12 questions) ──
    {
        'id': 28, 'category': 'symptom_diagnosis',
        'question': 'What does the FAST acronym stand for in stroke recognition?',
        'expected_answer': 'Face drooping, Arm weakness, Speech difficulty, Time to call 911',
        'relevant_doc_ids': ['Stroke Recognition and Acute Management'],
        'keywords': ['Face', 'Arm', 'Speech', 'Time']
    },
    {
        'id': 29, 'category': 'symptom_diagnosis',
        'question': 'What are the five life-threatening causes of chest pain?',
        'expected_answer': 'ACS, aortic dissection, pulmonary embolism, tension pneumothorax, esophageal rupture',
        'relevant_doc_ids': ['Chest Pain Differential Diagnosis and Evaluation'],
        'keywords': ['ACS', 'aortic dissection', 'pulmonary embolism', 'pneumothorax']
    },
    {
        'id': 30, 'category': 'symptom_diagnosis',
        'question': 'When does troponin I begin to rise after myocardial injury?',
        'expected_answer': '2-4 hours after myocardial injury, peaking at 12-24 hours',
        'relevant_doc_ids': ['Chest Pain Differential Diagnosis and Evaluation'],
        'keywords': ['2-4 hours', 'troponin', 'rise', '12-24']
    },
    {
        'id': 31, 'category': 'symptom_diagnosis',
        'question': 'What is the KDIGO definition of AKI based on serum creatinine?',
        'expected_answer': 'Increase ≥0.3 mg/dL within 48 hours or ≥1.5× baseline within 7 days',
        'relevant_doc_ids': ['Acute Kidney Injury (AKI) Diagnosis and Management'],
        'keywords': ['0.3 mg/dL', '48 hours', '1.5×', 'baseline']
    },
    {
        'id': 32, 'category': 'symptom_diagnosis',
        'question': 'What urine output criterion defines AKI in KDIGO?',
        'expected_answer': 'Urine output < 0.5 mL/kg/hr for ≥6 hours',
        'relevant_doc_ids': ['Acute Kidney Injury (AKI) Diagnosis and Management'],
        'keywords': ['0.5 mL/kg/hr', '6 hours', 'urine output']
    },
    {
        'id': 33, 'category': 'symptom_diagnosis',
        'question': 'What clinical features suggest community-acquired pneumonia?',
        'expected_answer': 'New pulmonary infiltrate plus 2 of: fever, leukocytosis, purulent sputum, new cough, pleuritic pain',
        'relevant_doc_ids': ['Pneumonia Diagnosis and Management'],
        'keywords': ['infiltrate', 'fever', 'cough', 'sputum']
    },
    {
        'id': 34, 'category': 'symptom_diagnosis',
        'question': 'What is the FeNa value that suggests pre-renal AKI?',
        'expected_answer': 'FeNa less than 1%',
        'relevant_doc_ids': ['Acute Kidney Injury (AKI) Diagnosis and Management'],
        'keywords': ['FeNa', '1%', 'pre-renal']
    },
    {
        'id': 35, 'category': 'symptom_diagnosis',
        'question': 'What clinical features suggest aortic dissection in a chest pain patient?',
        'expected_answer': 'Tearing/ripping pain radiating to back, hypertensive history, pulse differential, mediastinal widening on CXR',
        'relevant_doc_ids': ['Chest Pain Differential Diagnosis and Evaluation'],
        'keywords': ['tearing', 'back', 'pulse differential', 'mediastinal']
    },
    {
        'id': 36, 'category': 'symptom_diagnosis',
        'question': 'What sepsis definition involves organ dysfunction from infection?',
        'expected_answer': 'Life-threatening organ dysfunction caused by dysregulated host response to infection',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['organ dysfunction', 'dysregulated', 'host response', 'infection']
    },
    {
        'id': 37, 'category': 'symptom_diagnosis',
        'question': 'What Wells DVT score is considered high probability?',
        'expected_answer': 'Score 2 or higher',
        'relevant_doc_ids': ['Deep Vein Thrombosis (DVT) Treatment Protocol'],
        'keywords': ['Wells', '2', 'high probability', 'DVT']
    },
    {
        'id': 38, 'category': 'symptom_diagnosis',
        'question': 'What percentage of strokes are ischemic versus hemorrhagic?',
        'expected_answer': 'Ischemic: 87%, Hemorrhagic: 13%',
        'relevant_doc_ids': ['Stroke Recognition and Acute Management'],
        'keywords': ['87%', 'ischemic', '13%', 'hemorrhagic']
    },
    {
        'id': 39, 'category': 'symptom_diagnosis',
        'question': 'What is the HEART score threshold for high cardiac event risk in the ED?',
        'expected_answer': 'HEART score ≥4 indicates >13% major adverse cardiac event risk',
        'relevant_doc_ids': ['Chest Pain Differential Diagnosis and Evaluation'],
        'keywords': ['HEART', '4', '13%', 'high risk']
    },
    # ── Treatment Protocols (11 questions) ──
    {
        'id': 40, 'category': 'treatment_protocol',
        'question': 'What is the first-line vasopressor for septic shock?',
        'expected_answer': 'Norepinephrine',
        'relevant_doc_ids': ['Sepsis Management Guidelines (Surviving Sepsis Campaign)'],
        'keywords': ['norepinephrine', 'first-line', 'vasopressor']
    },
    {
        'id': 41, 'category': 'treatment_protocol',
        'question': 'What is the duration of DAPT recommended after STEMI with PCI?',
        'expected_answer': '12 months after PCI',
        'relevant_doc_ids': ['Acute MI (STEMI) Management Protocol'],
        'keywords': ['12 months', 'DAPT', 'PCI']
    },
    {
        'id': 42, 'category': 'treatment_protocol',
        'question': 'When is NIV (BiPAP) indicated in COPD exacerbation?',
        'expected_answer': 'Acute hypercapnic respiratory failure (PaCO2 > 45 with pH < 7.35)',
        'relevant_doc_ids': ['COPD Exacerbation Treatment Protocol'],
        'keywords': ['NIV', 'BiPAP', 'hypercapnic', 'pH']
    },
    {
        'id': 43, 'category': 'treatment_protocol',
        'question': 'What is the reversal agent for dabigatran?',
        'expected_answer': 'Idarucizumab',
        'relevant_doc_ids': ['Anticoagulation for Atrial Fibrillation'],
        'keywords': ['idarucizumab', 'dabigatran', 'reversal']
    },
    {
        'id': 44, 'category': 'treatment_protocol',
        'question': 'What are the indications for emergency dialysis using the AEIOU mnemonic?',
        'expected_answer': 'Acidosis, Electrolyte abnormalities, Ingestion, fluid Overload, Uremia',
        'relevant_doc_ids': ['Acute Kidney Injury (AKI) Diagnosis and Management'],
        'keywords': ['AEIOU', 'Acidosis', 'Uremia', 'dialysis']
    },
    {
        'id': 45, 'category': 'treatment_protocol',
        'question': 'What anticoagulant treatment is preferred for cancer-associated DVT?',
        'expected_answer': 'LMWH or rivaroxaban/edoxaban; indefinite treatment while cancer is active',
        'relevant_doc_ids': ['Deep Vein Thrombosis (DVT) Treatment Protocol'],
        'keywords': ['LMWH', 'cancer', 'DVT', 'rivaroxaban']
    },
    {
        'id': 46, 'category': 'treatment_protocol',
        'question': 'What is the morphine IV dose used for acute pain management?',
        'expected_answer': '2-4 mg IV every 3-4 hours',
        'relevant_doc_ids': ['Acute Pain Management Protocol'],
        'keywords': ['2-4 mg', 'morphine', 'IV', '3-4 hours']
    },
    {
        'id': 47, 'category': 'treatment_protocol',
        'question': 'What intravenous antihypertensives are used for hypertensive emergency?',
        'expected_answer': 'IV labetalol, nicardipine, or hydralazine',
        'relevant_doc_ids': ['Hypertension Management Guidelines (JNC 8 / ACC/AHA 2017)'],
        'keywords': ['labetalol', 'nicardipine', 'hydralazine', 'hypertensive emergency']
    },
    {
        'id': 48, 'category': 'treatment_protocol',
        'question': 'What is the initial rivaroxaban dose for acute DVT treatment?',
        'expected_answer': '15 mg twice daily for 21 days, then 20 mg daily',
        'relevant_doc_ids': ['Deep Vein Thrombosis (DVT) Treatment Protocol'],
        'keywords': ['15 mg', 'rivaroxaban', '21 days', '20 mg']
    },
    {
        'id': 49, 'category': 'treatment_protocol',
        'question': 'What is the H. pylori eradication triple therapy regimen?',
        'expected_answer': 'Amoxicillin 1000 mg + clarithromycin 500 mg + PPI twice daily for 14 days',
        'relevant_doc_ids': ['Amoxicillin Antibiotic Reference'],
        'keywords': ['amoxicillin', 'clarithromycin', 'PPI', '14 days']
    },
    {
        'id': 50, 'category': 'treatment_protocol',
        'question': 'What post-MI medications are recommended after PCI for STEMI?',
        'expected_answer': 'Beta-blocker (metoprolol), ACE inhibitor (lisinopril), high-intensity statin (atorvastatin 80 mg)',
        'relevant_doc_ids': ['Acute MI (STEMI) Management Protocol'],
        'keywords': ['beta-blocker', 'ACE inhibitor', 'statin', 'atorvastatin 80 mg']
    }
]

eval_df = pd.DataFrame(EVAL_DATASET)
print(f'✅ Evaluation dataset created: {len(EVAL_DATASET)} question-answer pairs')
print(f'\nCategory distribution:')
print(eval_df['category'].value_counts().to_string())

## 📐 Section 6: Metric Computation Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# RETRIEVAL METRICS
# ─────────────────────────────────────────────────────────────────────

def compute_recall_at_k(retrieved_doc_titles: List[str],
                         relevant_doc_titles: List[str],
                         k: int) -> float:
    """
    Compute Recall@K for a single query.
    
    Formula: Recall@K = |{relevant docs in top-K retrieved}| / |{total relevant docs}|
    
    Args:
        retrieved_doc_titles: List of document titles retrieved (in rank order)
        relevant_doc_titles: Ground truth list of relevant document titles
        k: Cutoff rank
    Returns:
        Recall@K score [0.0, 1.0]
    """
    top_k = retrieved_doc_titles[:k]
    # Case-insensitive partial matching (chunk titles may not exactly match)
    hits = sum(
        1 for rel_title in relevant_doc_titles
        if any(rel_title.lower() in ret.lower() or ret.lower() in rel_title.lower()
               for ret in top_k)
    )
    return hits / max(len(relevant_doc_titles), 1)


def compute_mrr(retrieved_doc_titles: List[str],
                relevant_doc_titles: List[str]) -> float:
    """
    Compute Mean Reciprocal Rank (MRR) for a single query.
    
    Formula: MRR = 1 / rank_of_first_relevant_document
    Returns 0 if no relevant document found in retrieved list.
    
    Args:
        retrieved_doc_titles: List of retrieved document titles (in rank order)
        relevant_doc_titles: Ground truth relevant document titles
    Returns:
        Reciprocal rank score [0.0, 1.0]
    """
    for rank, ret_title in enumerate(retrieved_doc_titles, start=1):
        if any(rel.lower() in ret_title.lower() or ret_title.lower() in rel.lower()
               for rel in relevant_doc_titles):
            return 1.0 / rank
    return 0.0


# ─────────────────────────────────────────────────────────────────────
# GENERATION METRICS
# ─────────────────────────────────────────────────────────────────────

def compute_faithfulness_score(answer: str, context_docs: List[Document]) -> float:
    """
    Compute Faithfulness Score using LLM-as-judge.
    
    Formula: Faithfulness = grounded_claims / total_claims
    Uses Gemini to extract claims from the answer and verify each against context.
    
    Args:
        answer: Generated answer string
        context_docs: List of retrieved context documents
    Returns:
        Faithfulness score [0.0, 1.0]
    """
    context_str = ' '.join([d.page_content for d in context_docs[:3]])[:2000]
    
    prompt = f"""
Extract all factual claims from this medical answer and verify each against the context.

Answer: {answer[:800]}

Context: {context_str}

Respond ONLY with JSON:
{{"total_claims": N, "grounded_claims": N, "faithfulness_score": 0.0-1.0}}
"""
    try:
        result = llm.invoke(prompt).content
        result_clean = result.replace('```json', '').replace('```', '').strip()
        data = json.loads(result_clean)
        return float(data.get('faithfulness_score', 0.5))
    except:
        # Keyword overlap fallback
        answer_words = set(answer.lower().split())
        context_words = set(context_str.lower().split())
        overlap = len(answer_words & context_words)
        return min(overlap / max(len(answer_words), 1), 1.0)


def compute_answer_relevance(question: str, answer: str) -> float:
    """
    Compute Answer Relevance as cosine similarity between question and answer embeddings.
    
    Formula: relevance = cosine_similarity(embed(question), embed(answer))
    Uses normalized embeddings from the SentenceTransformer model.
    
    Args:
        question: Original clinical question
        answer: Generated answer string
    Returns:
        Cosine similarity score [0.0, 1.0]
    """
    q_emb = np.array(embedding_model.embed_query(question)).reshape(1, -1)
    a_emb = np.array(embedding_model.embed_query(answer[:500])).reshape(1, -1)
    sim = cosine_similarity(q_emb, a_emb)[0][0]
    return float(max(sim, 0.0))  # Clamp to [0, 1]


def compute_hallucination_rate_batch(answers: List[str],
                                      context_docs_list: List[List[Document]]) -> float:
    """
    Compute Hallucination Rate across a batch of answers.
    
    Formula: Hallucination Rate = answers_with_ungrounded_claims / total_answers
    An answer is hallucinated if it contains specific claims (numbers, drug names,
    thresholds) not present in the context.
    
    Args:
        answers: List of generated answers
        context_docs_list: Corresponding list of context documents per answer
    Returns:
        Hallucination rate [0.0, 1.0] (lower is better)
    """
    hallucinated_count = 0
    for answer, docs in zip(answers, context_docs_list):
        context_text = ' '.join([d.page_content for d in docs]).lower()
        # Extract numeric/specific claims from answer using simple regex
        specific_claims = re.findall(r'\b\d+\.?\d*\s*(?:mg|mL|mmHg|%|hours|days|weeks)\b',
                                      answer.lower())
        if not specific_claims:
            continue  # No verifiable numeric claims
        ungrounded = sum(1 for claim in specific_claims
                         if claim.split()[0] not in context_text)
        if ungrounded > len(specific_claims) * 0.3:  # >30% claims unverifiable
            hallucinated_count += 1
    return hallucinated_count / max(len(answers), 1)


def compute_groundedness(answer: str, context_docs: List[Document]) -> Dict[str, Any]:
    """
    Detailed groundedness analysis: extract claims and verify each against source docs.
    
    Formula: Groundedness = grounded_claims / total_claims
    
    Args:
        answer: Generated answer string
        context_docs: Retrieved source documents
    Returns:
        Dict with: total_claims, grounded_claims, groundedness_score, ungrounded_claims
    """
    context_str = ' '.join([d.page_content for d in context_docs])[:3000]
    
    prompt = f"""
List each specific factual claim in the answer below, then check if each claim
is supported by the context. Focus on medical facts (dosages, thresholds, diagnoses).

Answer: {answer[:600]}
Context: {context_str[:1500]}

Respond ONLY with JSON:
{{
  "claims": [{{"claim": "...", "grounded": true/false}}],
  "total_claims": N,
  "grounded_claims": N,
  "groundedness_score": 0.0-1.0
}}
"""
    try:
        result = llm.invoke(prompt).content
        result_clean = result.replace('```json', '').replace('```', '').strip()
        data = json.loads(result_clean)
        return {
            'total_claims': data.get('total_claims', 0),
            'grounded_claims': data.get('grounded_claims', 0),
            'groundedness_score': data.get('groundedness_score', 0.5),
            'ungrounded_claims': [c['claim'] for c in data.get('claims', [])
                                   if not c.get('grounded', True)]
        }
    except:
        return {'total_claims': 1, 'grounded_claims': 1, 'groundedness_score': 0.5, 'ungrounded_claims': []}


print('✅ All metric functions defined')

## ⚖️ Section 7: Baseline RAG vs CRAG Comparison

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# NAIVE RAG PIPELINE (baseline for comparison)
# ─────────────────────────────────────────────────────────────────────

NAIVE_GENERATE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'You are a medical AI assistant. Answer the question based on the context provided.'),
    ('human', 'Question: {question}\n\nContext:\n{context}\n\nAnswer:')
])

naive_rag_chain = (
    {'question': RunnablePassthrough(),
     'context': lambda x: '\n\n'.join([d.page_content for d in retriever.get_relevant_documents(x)])}
    | NAIVE_GENERATE_PROMPT
    | llm
    | StrOutputParser()
)


def run_naive_rag(question: str) -> Dict[str, Any]:
    """Run question through naive RAG (retrieve → generate, no grading or correction)."""
    docs = retriever.get_relevant_documents(question)
    answer = naive_rag_chain.invoke(question)
    return {
        'question': question,
        'answer': answer,
        'documents': docs,
        'sources': [d.metadata.get('title', 'Unknown') for d in docs]
    }


print('✅ Naive RAG pipeline defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# RUN EVALUATION ON SUBSET (use all 50 or subset for speed)
# ─────────────────────────────────────────────────────────────────────

# For quick demo, evaluate first 10 questions. For full eval, set EVAL_SUBSET = None
EVAL_SUBSET = 10  # Set to None to run all 50
eval_subset = EVAL_DATASET[:EVAL_SUBSET] if EVAL_SUBSET else EVAL_DATASET

print(f'Running evaluation on {len(eval_subset)} questions...')
print('This will take a few minutes due to LLM calls.\n')

naive_results = []
crag_results = []

for i, qa in enumerate(eval_subset):
    print(f'[{i+1}/{len(eval_subset)}] {qa["question"][:60]}...')
    
    # ── Naive RAG ──
    try:
        naive_r = run_naive_rag(qa['question'])
        naive_retrieved_titles = [d.metadata.get('title', '') for d in naive_r['documents']]
        naive_results.append({
            'id': qa['id'],
            'question': qa['question'],
            'expected': qa['expected_answer'],
            'answer': naive_r['answer'],
            'documents': naive_r['documents'],
            'retrieved_titles': naive_retrieved_titles,
            'recall_at_3': compute_recall_at_k(naive_retrieved_titles, qa['relevant_doc_ids'], 3),
            'recall_at_5': compute_recall_at_k(naive_retrieved_titles, qa['relevant_doc_ids'], 5),
            'mrr': compute_mrr(naive_retrieved_titles, qa['relevant_doc_ids']),
            'answer_relevance': compute_answer_relevance(qa['question'], naive_r['answer']),
            'faithfulness': compute_faithfulness_score(naive_r['answer'], naive_r['documents'])
        })
    except Exception as e:
        print(f'  ⚠️  Naive RAG error: {e}')
    
    # ── CRAG ──
    try:
        crag_r = run_crag(qa['question'], verbose=False)
        crag_retrieved_titles = [d.metadata.get('title', '') for d in crag_r['documents']]
        crag_results.append({
            'id': qa['id'],
            'question': qa['question'],
            'expected': qa['expected_answer'],
            'answer': crag_r['answer'],
            'documents': crag_r['documents'],
            'retrieved_titles': crag_retrieved_titles,
            'pipeline_path': ' → '.join(crag_r['pipeline_path']),
            'hallucination_check': crag_r['hallucination_check'],
            'recall_at_3': compute_recall_at_k(crag_retrieved_titles, qa['relevant_doc_ids'], 3),
            'recall_at_5': compute_recall_at_k(crag_retrieved_titles, qa['relevant_doc_ids'], 5),
            'mrr': compute_mrr(crag_retrieved_titles, qa['relevant_doc_ids']),
            'answer_relevance': compute_answer_relevance(qa['question'], crag_r['answer']),
            'faithfulness': compute_faithfulness_score(crag_r['answer'], crag_r['documents'])
        })
    except Exception as e:
        print(f'  ⚠️  CRAG error: {e}')

print(f'\n✅ Evaluation complete: {len(naive_results)} naive / {len(crag_results)} CRAG results')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# COMPUTE AGGREGATE METRICS
# ─────────────────────────────────────────────────────────────────────

def aggregate_metrics(results: List[Dict]) -> Dict[str, float]:
    """Compute mean metrics across all evaluation results."""
    if not results:
        return {}
    return {
        'recall_at_3': np.mean([r['recall_at_3'] for r in results]),
        'recall_at_5': np.mean([r['recall_at_5'] for r in results]),
        'mrr': np.mean([r['mrr'] for r in results]),
        'faithfulness': np.mean([r['faithfulness'] for r in results]),
        'answer_relevance': np.mean([r['answer_relevance'] for r in results]),
        'hallucination_rate': compute_hallucination_rate_batch(
            [r['answer'] for r in results],
            [r['documents'] for r in results]
        )
    }


naive_metrics = aggregate_metrics(naive_results)
crag_metrics = aggregate_metrics(crag_results)

print('✅ Aggregate metrics computed')
print(f'\nNaive RAG: {naive_metrics}')
print(f'CRAG:      {crag_metrics}')

## 📈 Section 8: Results & Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FINAL SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────

def format_improvement(naive_val: float, crag_val: float, higher_is_better: bool = True) -> str:
    """Format improvement metric with sign and color indication."""
    delta = crag_val - naive_val
    sign = '+' if delta >= 0 else ''
    return f'{sign}{delta:.1%}' if abs(delta) < 1 else f'{sign}{delta:.3f}'


summary_data = [
    {
        'Metric': 'Recall@3',
        'Naive RAG': f"{naive_metrics.get('recall_at_3', 0):.1%}",
        'CRAG': f"{crag_metrics.get('recall_at_3', 0):.1%}",
        'Improvement': format_improvement(naive_metrics.get('recall_at_3', 0), crag_metrics.get('recall_at_3', 0)),
        'Higher Better': '✅'
    },
    {
        'Metric': 'Recall@5',
        'Naive RAG': f"{naive_metrics.get('recall_at_5', 0):.1%}",
        'CRAG': f"{crag_metrics.get('recall_at_5', 0):.1%}",
        'Improvement': format_improvement(naive_metrics.get('recall_at_5', 0), crag_metrics.get('recall_at_5', 0)),
        'Higher Better': '✅'
    },
    {
        'Metric': 'MRR',
        'Naive RAG': f"{naive_metrics.get('mrr', 0):.3f}",
        'CRAG': f"{crag_metrics.get('mrr', 0):.3f}",
        'Improvement': format_improvement(naive_metrics.get('mrr', 0), crag_metrics.get('mrr', 0)),
        'Higher Better': '✅'
    },
    {
        'Metric': 'Faithfulness',
        'Naive RAG': f"{naive_metrics.get('faithfulness', 0):.1%}",
        'CRAG': f"{crag_metrics.get('faithfulness', 0):.1%}",
        'Improvement': format_improvement(naive_metrics.get('faithfulness', 0), crag_metrics.get('faithfulness', 0)),
        'Higher Better': '✅'
    },
    {
        'Metric': 'Hallucination Rate',
        'Naive RAG': f"{naive_metrics.get('hallucination_rate', 0):.1%}",
        'CRAG': f"{crag_metrics.get('hallucination_rate', 0):.1%}",
        'Improvement': format_improvement(crag_metrics.get('hallucination_rate', 0), naive_metrics.get('hallucination_rate', 0)),
        'Higher Better': '❌ (lower better)'
    },
    {
        'Metric': 'Answer Relevance',
        'Naive RAG': f"{naive_metrics.get('answer_relevance', 0):.3f}",
        'CRAG': f"{crag_metrics.get('answer_relevance', 0):.3f}",
        'Improvement': format_improvement(naive_metrics.get('answer_relevance', 0), crag_metrics.get('answer_relevance', 0)),
        'Higher Better': '✅'
    }
]

summary_df = pd.DataFrame(summary_data)

print('\n' + '='*75)
print('         MEDICAL CRAG EVALUATION RESULTS — NAIVE RAG vs CRAG')
print('='*75)
print(summary_df.to_string(index=False))
print('='*75)
print(f'  Evaluation set: {len(eval_subset)} questions | Model: Gemini 1.5 Flash')
print(f'  Embeddings: all-MiniLM-L6-v2 | Vector DB: ChromaDB')
print('='*75)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# SAVE RESULTS TO CSV (for resume metrics)
# ─────────────────────────────────────────────────────────────────────

# Detailed per-question results
naive_df = pd.DataFrame([{
    'id': r['id'], 'category': eval_subset[i]['category'],
    'question': r['question'][:80],
    'recall_at_3': r['recall_at_3'], 'recall_at_5': r['recall_at_5'],
    'mrr': r['mrr'], 'faithfulness': r['faithfulness'],
    'answer_relevance': r['answer_relevance'],
    'pipeline': 'Naive RAG'
} for i, r in enumerate(naive_results)])

crag_df_detail = pd.DataFrame([{
    'id': r['id'], 'category': eval_subset[i]['category'],
    'question': r['question'][:80],
    'recall_at_3': r['recall_at_3'], 'recall_at_5': r['recall_at_5'],
    'mrr': r['mrr'], 'faithfulness': r['faithfulness'],
    'answer_relevance': r['answer_relevance'],
    'pipeline_path': r['pipeline_path'],
    'hallucination_check': r['hallucination_check'],
    'pipeline': 'CRAG'
} for i, r in enumerate(crag_results)])

combined_df = pd.concat([naive_df, crag_df_detail], ignore_index=True)
combined_df.to_csv('crag_evaluation_results.csv', index=False)
summary_df.to_csv('crag_summary_metrics.csv', index=False)

print('✅ Saved: crag_evaluation_results.csv')
print('✅ Saved: crag_summary_metrics.csv')
print(f'\nFinal Resume Metrics:')
print(f'  Recall@5 (CRAG):        {crag_metrics.get("recall_at_5", 0):.1%}')
print(f'  MRR (CRAG):             {crag_metrics.get("mrr", 0):.3f}')
print(f'  Faithfulness (CRAG):    {crag_metrics.get("faithfulness", 0):.1%}')
print(f'  Hallucination (CRAG):   {crag_metrics.get("hallucination_rate", 0):.1%}')
print(f'  Answer Relevance (CRAG): {crag_metrics.get("answer_relevance", 0):.3f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# PIPELINE PATH ANALYSIS
# ─────────────────────────────────────────────────────────────────────
if crag_results:
    paths = [r['pipeline_path'] for r in crag_results]
    path_counts = pd.Series(paths).value_counts()
    
    print('\nCRAG Pipeline Path Distribution:')
    print(path_counts.to_string())
    
    grounded_count = sum(1 for r in crag_results if r['hallucination_check'] == 'grounded')
    print(f'\nGrounded answers: {grounded_count}/{len(crag_results)} ({grounded_count/len(crag_results):.1%})')